In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Run in collab
<a href="https://colab.research.google.com/github/racousin/rl_introduction/blob/master/notebooks/5_policy_gradient-reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym

In [ ]:
# We will experiment our algo with CartPole
env = gym.make('CartPole-v0')

# module13_exercise5: Policy Optimization

### Objective
Here we present an alternative of Q learning: policy gradient algorithm

**Complete the TODO steps! Good luck!**

# Policy gradient
In policy gradient, we parametrize directly the policy $\pi_\theta$. It's especially welcome when the action space is continuous; in that case greedy policy based on Q-learning need to compute the $argmax_a Q(s,a)$. This could be pretty tedious. More generally, policy gradient algorithms are better to explore large state-action spaces.

$J(\pi_{\theta}) = E_{\tau \sim \pi_{\theta}}[{G(\tau)}]$

We can proof  that:


$\nabla_{\theta} J(\pi_{\theta}) = E_{\tau \sim \pi_{\theta}}[{\sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_t |s_t) G(\tau)}]$ 

1. In discrete action space

we parametrize $\pi$ with $\theta$, such as $\pi_\theta : S \rightarrow [0,1]^{dim(A)}$ and $\forall s$ $\sum \pi_\theta(s) = 1$.


2. In continous action space

we parametrize $\pi$ with $\theta$, such as $\pi_\theta : S \rightarrow \mu^{dim(A)} \times \sigma^{dim(A)} =  \mathbb{R}^{dim(A)} \times \mathbb{R}_{+,*}^{dim(A)}$



In torch, it is easier to pass the loss than the gradient.
1. It is possible to show that the loss for discrete action ($1,...,N$) with softmax policy is weighted negative binary crossentropy:
$-G\sum_{j=1}^N[a^j\log(\hat{a}^j) + (1-a^j)\log(1 - \hat{a}^j)]$

with:
$a^j=1$ if $a_t = j$, $0$ otherwise.

$\hat{a}^j = \pi_\theta(s_t)^j$.

$G$ is the discounted empirical return $G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$ from state $s_t$ and $a_t$


2. It is possible to show that the loss for conitnous action ($1,...,N$) with multivariate Gaussian (identity Covariance) policy is given by:

$-G\sum_{j=1}^N[(a^j - \hat{a}^j)^2]$

$\hat{a}^j = \pi_\theta(s_t)^j$.



see https://aleksispi.github.io/assets/pg_autodiff.pdf for more explanation

# Reinforce - discrete action

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt

### TODO 0): write policy gradient interaction with the environment

In [ ]:
#Done: write a keras model that represent our parametrized pi function
# We should be able to run pi.predict([s]) and it should return [[P(a_0|s), P(a_1|s) .. P(a_m|s)]] where m is action size
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Tuple, List

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, action_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=-1)
        return x

In [ ]:
#TODO: write the action choosen by our initial policy gradient function.
# It should be a ~ P(.|s) = U(pi_fonction(s))
class DeepAgent:
    def __init__(self, env, gamma: float = 0.99, epsilon: float = 0.01):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.action_dim = env.action_space.n

class ReinforceAgent(DeepAgent):
    def __init__(self, env, policy_network: nn.Module, gamma: float = 0.99, epsilon: float = 0.01):
        super().__init__(env, gamma, epsilon)
        self.policy = policy_network
        self.policy.eval()  # Set to evaluation mode
        self.device = next(self.policy.parameters()).device
        self.episode: List = []

    def act(self, state: np.ndarray) -> int:
        # Convert state to torch tensor
        state_tensor = torch.FloatTensor(state).reshape(1, -1).to(self.device)
        
        # Get action probabilities
        with torch.no_grad():
            probs = self.policy(state_tensor).cpu().numpy().flatten()
        
        # Sample action from probability distribution
        action = np.random.choice(self.action_dim, p=probs)
        return action

In [ ]:
#Done: write the action choosen by our initial policy gradient function.
# It should be a ~ P(.|s) = U(pi_fonction(s))
class ReinforceAgent(DeepAgent):
    def __init__(self, env, compiled_model, gamma = .99, epsilon = .01):
        super().__init__(env,gamma, epsilon)
        
        self.model = compiled_model
        self.model.summary()
        
        self.episode = []

    def act(self, state):
        state = state.reshape(1, -1)
        prob = self.model.predict(state, batch_size=1, verbose=0).flatten()
        action = np.random.choice(self.action_dim, 1, p=prob)[0]
        return action

In [ ]:
def run_experiment_episode(env, agent: ReinforceAgent, nb_episode: int) -> np.ndarray:
    rewards = np.zeros(nb_episode)
    
    for i in range(nb_episode):
        state, _ = env.reset()
        done = False
        episode_rewards = []
        
        while not done:
            action = agent.act(state)
            state, reward, terminated, truncated, info = env.step(action)
            episode_rewards.append(reward)
            done = terminated or truncated
        rewards[i] = sum(episode_rewards)
        print(f'episode: {i} - cum reward {rewards[i]}')
    
    return rewards

In [ ]:
#TODO: interact with the environment through episode and display the return

In [ ]:
def setup_experiment(env):
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    # Create policy network
    policy_network = PolicyNetwork(state_dim, action_dim)
    
    # Create agent
    agent = ReinforceAgent(env, policy_network)
    
    return agent

# Visualization function
def plot_rewards(rewards: np.ndarray, title: str = 'Cumulative Reward per Episode'):
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.plot(rewards, '+')
    ax.set_title(title)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Cumulative Reward')
    plt.show()

In [ ]:
# Initialize environment
import gym
env = gym.make('CartPole-v1')

# Setup agent and run experiment
agent = setup_experiment(env)
rewards = run_experiment_episode(env, agent, 20)

# Plot results
plot_rewards(rewards)

### TODO 1): write custom loss for policy gradient

weighted negative binary crossentropy:
$-G\sum_{j=1}^N[a^j\log(\hat{a}^j) + (1-a^j)\log(1 - \hat{a}^j)]$

In [ ]:
#TODO: write custom loss for policy gradient
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, action_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=-1)
        return x

### TODO 2): complete training of vanilla policy gradient

In [ ]:
#TODO: complete training of vanilla policy gradient
# 
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Tuple
import gym
from pathlib import Path

def discount_cumsum(rewards: np.ndarray, gamma: float) -> np.ndarray:
    """Calculate discounted cumulative returns."""
    returns = np.zeros_like(rewards)
    running_sum = 0
    for t in reversed(range(len(rewards))):
        running_sum = rewards[t] + gamma * running_sum
        returns[t] = running_sum
    return returns

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, action_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=-1)
        return x

class DeepAgent:
    def __init__(self, env: gym.Env, gamma: float = 0.99, epsilon: float = 0.01):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.action_dim = env.action_space.n

class ReinforceAgent(DeepAgent):
    def __init__(
        self, 
        env: gym.Env, 
        policy: PolicyNetwork, 
        learning_rate: float = 1e-3,
        gamma: float = 0.99, 
        epsilon: float = 0.01,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        super().__init__(env, gamma, epsilon)
        self.device = device
        self.policy = policy.to(device)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=learning_rate)
        self.episode: List = []

    def act(self, state: np.ndarray) -> int:
        """Select action based on policy."""
        state_tensor = torch.FloatTensor(state).reshape(1, -1).to(self.device)
        with torch.no_grad():
            probs = self.policy(state_tensor).cpu().numpy().flatten()
        action = np.random.choice(self.action_dim, p=probs)
        return action

    def train(self, current_state: np.ndarray, action: int, reward: float, 
              next_state: np.ndarray, done: bool) -> float:
        """Train the policy on a single transition."""
        self.episode.append((current_state, action, reward))
        
        if done:
            # Process episode data
            episode = np.array(self.episode)
            states = np.vstack([x[0] for x in self.episode])
            actions = np.array([x[1] for x in self.episode])
            rewards = np.array([x[2] for x in self.episode])
            
            # Calculate discounted returns
            returns = discount_cumsum(rewards, self.gamma)
            
            # Convert to tensors and move to device
            states_tensor = torch.FloatTensor(states).to(self.device)
            actions_tensor = torch.LongTensor(actions).to(self.device)
            returns_tensor = torch.FloatTensor(returns).to(self.device)
            
            # Get action probabilities
            self.policy.train()
            action_probs = self.policy(states_tensor)
            
            # Calculate log probabilities of taken actions
            action_log_probs = torch.log(action_probs[torch.arange(len(actions)), actions])
            
            # Calculate loss
            loss = -(returns_tensor * action_log_probs).mean()
            
            # Optimize
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            # Reset episode buffer
            self.episode = []
            self.policy.eval()
            
            return loss.item()
        return 0.0

    def save_model(self, path: str):
        """Save the policy network."""
        torch.save({
            'policy_state_dict': self.policy.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, path)

    def load_model(self, path: str):
        """Load the policy network."""
        checkpoint = torch.load(path)
        self.policy.load_state_dict(checkpoint['policy_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.policy.eval()

# Example usage
def train_agent(env_name: str = 'CartPole-v1', num_episodes: int = 1000):
    # Create environment
    env = gym.make(env_name)
    
    # Initialize agent
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    policy = PolicyNetwork(state_dim, action_dim)
    agent = ReinforceAgent(env, policy)
    
    # Training loop
    episode_rewards = []
    for episode in range(num_episodes):
        state = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            action = agent.act(state)
            next_state, reward, done, _ = env.step(action)
            loss = agent.train(state, action, reward, next_state, done)
            episode_reward += reward
            state = next_state
            
        episode_rewards.append(episode_reward)
        if episode % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f'Episode {episode}, Average Reward: {avg_reward:.2f}')
    
    return agent, episode_rewards

if __name__ == "__main__":
    agent, rewards = train_agent()
    
    # Plot results
    import matplotlib.pyplot as plt
    plt.plot(rewards)
    plt.title('Training Rewards')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.show()

In [ ]:
# Initialize environment
import gym
env = gym.make('CartPole-v1')

# Setup agent and run experiment
agent = setup_experiment(env)
rewards = run_experiment_episode(env, agent, 20)

# Plot results
plot_rewards(rewards)

### TODO 3) : Try different hyerparamters models (number of layers, nodes) and compare learning

# Reinforce with memory - discrete action

In opposite as Q learning, policy optimization is an on-policy algorithm, so we are training directly on the policy output and we need to compute them first.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Tuple, Optional
import gym
from pathlib import Path

def discount_cumsum(rewards: np.ndarray, gamma: float) -> np.ndarray:
    """Calculate discounted cumulative returns."""
    returns = np.zeros_like(rewards)
    running_sum = 0
    for t in reversed(range(len(rewards))):
        running_sum = rewards[t] + gamma * running_sum
        returns[t] = running_sum
    return returns

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, action_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=-1)
        return x

class DeepAgent:
    def __init__(self, env: gym.Env, gamma: float = 0.99, epsilon: float = 0.01):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.action_dim = env.action_space.n

class ReinforceAgentWithMemory(DeepAgent):
    def __init__(
        self, 
        env: gym.Env, 
        policy: PolicyNetwork,
        memory_size: int = 3,
        learning_rate: float = 1e-2,
        gamma: float = 0.99, 
        epsilon: float = 0.01,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        super().__init__(env, gamma, epsilon)
        self.device = device
        self.policy = policy.to(device)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=learning_rate)
        
        # Memory management
        self.memory_size = memory_size
        self.episodes: List = []  # Store complete episodes
        self.current_episode: List = []  # Store current episode transitions
        
    def act(self, state: np.ndarray) -> int:
        """Select action based on policy."""
        state_tensor = torch.FloatTensor(state).reshape(1, -1).to(self.device)
        with torch.no_grad():
            probs = self.policy(state_tensor).cpu().numpy().flatten()
        action = np.random.choice(self.action_dim, p=probs)
        return action

    def train(self, current_state: np.ndarray, action: int, reward: float, 
              next_state: np.ndarray, done: bool) -> Optional[float]:
        """Train the policy using episode memory."""
        # Store transition
        self.current_episode.append((current_state, action, reward))
        
        if done:
            # Process current episode
            episode_array = np.array(self.current_episode)
            states = np.vstack([x[0] for x in self.current_episode])
            actions = np.array([x[1] for x in self.current_episode])
            rewards = np.array([x[2] for x in self.current_episode])
            
            # Calculate discounted returns
            discounted_returns = discount_cumsum(rewards, self.gamma)
            
            # Store processed episode
            self.episodes.append((states, actions, discounted_returns))
            self.current_episode = []
            
            # Train if memory is full
            if len(self.episodes) == self.memory_size:
                return self._train_on_memory()
                
        return None

    def _train_on_memory(self) -> float:
        """Train on collected episodes when memory is full."""
        # Combine all episodes
        all_states = np.vstack([ep[0] for ep in self.episodes])
        all_actions = np.concatenate([ep[1] for ep in self.episodes])
        all_returns = np.concatenate([ep[2] for ep in self.episodes])
        
        # Normalize returns
        all_returns = (all_returns - all_returns.mean()) / (all_returns.std() + 1e-8)
        
        # Convert to tensors and move to device
        states_tensor = torch.FloatTensor(all_states).to(self.device)
        actions_tensor = torch.LongTensor(all_actions).to(self.device)
        returns_tensor = torch.FloatTensor(all_returns).to(self.device)
        
        # Get action probabilities
        self.policy.train()
        action_probs = self.policy(states_tensor)
        
        # Calculate log probabilities of taken actions
        action_log_probs = torch.log(action_probs[torch.arange(len(actions_tensor)), actions_tensor])
        
        # Calculate loss
        loss = -(returns_tensor * action_log_probs).mean()
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Reset memory
        self.episodes = []
        self.policy.eval()
        
        return loss.item()

    def save_model(self, path: str):
        """Save the policy network and training state."""
        torch.save({
            'policy_state_dict': self.policy.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, path)

    def load_model(self, path: str):
        """Load the policy network and training state."""
        checkpoint = torch.load(path)
        self.policy.load_state_dict(checkpoint['policy_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.policy.eval()

# Example usage and training loop
def train_agent(env_name: str = 'CartPole-v1', num_episodes: int = 1000, memory_size: int = 3):
    # Create environment
    env = gym.make(env_name)
    
    # Initialize agent
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    policy = PolicyNetwork(state_dim, action_dim)
    agent = ReinforceAgentWithMemory(env, policy, memory_size=memory_size)
    
    # Training loop
    episode_rewards = []
    for episode in range(num_episodes):
        state = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            action = agent.act(state)
            next_state, reward, done, _ = env.step(action)
            loss = agent.train(state, action, reward, next_state, done)
            episode_reward += reward
            state = next_state
            
            # Print training info when batch training occurs
            if loss is not None:
                print(f'Episode {episode}, Loss: {loss:.4f}')
        
        episode_rewards.append(episode_reward)
        if episode % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f'Episode {episode}, Average Reward: {avg_reward:.2f}')
    
    return agent, episode_rewards

if __name__ == "__main__":
    agent, rewards = train_agent()
    
    # Plot training results
    import matplotlib.pyplot as plt
    plt.plot(rewards)
    plt.title('Training Rewards')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.show()

In [ ]:
# Initialize environment
import gym
env = gym.make('CartPole-v1')

# Setup agent and run experiment
agent = setup_experiment(env)
rewards = run_experiment_episode(env, agent, 20)

# Plot results
plot_rewards(rewards)

# other improvements 

### GAE(general advantage estimation) actor critic
We can rewrite the policy gradient

$\nabla_{\theta} J(\pi_{\theta}) = E_{\tau \sim \pi_{\theta}}[{\sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_t |s_t) \Phi_t}]$,

whith $\Phi_t$ could be any of:
- $\Phi_t =  G_t$
- $\Phi_t = \sum_{t'=t}^T R_{t+1} - V(s_t)$
- $\Phi_t = \sum_{t'=t}^T R_{t+1} - Q(s_t,a_t)$


For the last 2 cases we need to estimate V or Q (the critics). We do it as the same way at deepQ.
https://arxiv.org/pdf/1506.02438.pdf

$\phi_k = \arg \min_{\phi} E_{s_t, G_t \sim \pi_k}[{\left( V_{\phi}(s_t) - G_t \right)^2}]$

### off policy
To build an experience replay for policy gradient, it is necessary to unbias the experiences.
https://arxiv.org/pdf/1205.4839.pdf

### clipping